# Notebook 2B — Unlearning · Evaluation · Results (Group B)

**Paper:** *An Illusion of Unlearning? Assessing Machine Unlearning Through Internal Representations*  
(Gao, Unal, Rangamani, Zhu — AISTATS 2026)

**Prerequisite:** Run **Notebook 1** first and attach its output as a Kaggle dataset.  
Then set `CKPT_DATASET_DIR` in Section A to the dataset mount path.

**This notebook runs Group B (5 methods). Run `notebook_2_unlearn` for Group A.**

| Stage | Description |
|-------|-------------|
| **A** | Environment setup, repo clone, load config from Notebook 1 |
| **B** | Group B method selection (5 methods) |
| **C** | Run Group B unlearning methods |
| **D** | Evaluation — Output acc · Linear Probe · NCC |
| **E** | Results table (Table 1 & 3 of paper) |
| **F** | Bar chart |
| **G** | t-SNE visualisation |

> **Recommended:** GPU T4/P100.  
> Full mode (5 methods × 15 forget groups) ≈ 2–3 h on T4.  
> `TEST_MODE` in `config.json` is inherited automatically from Notebook 1.

## A. Environment Setup & Load Config

In [ ]:
import subprocess, sys

def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout:
        print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr:
        print('STDERR:', r.stderr[-2000:])
    return r.returncode

sh('pip install -q timm einops scikit-learn matplotlib seaborn')

In [ ]:
import os, sys, json, copy, random, argparse, collections, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'

if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  SET THIS to the Kaggle dataset mount path exported from Notebook 1.
#  This notebook supports both common layouts:
#    1) <dataset>/config.json
#    2) <dataset>/checkpoints/config.json
#
#  Typical paths after attaching the dataset:
#    /kaggle/input/<your-dataset-slug>/
#    /kaggle/input/datasets/<owner>/<your-dataset-slug>/
# ══════════════════════════════════════════════════════════════════════
CKPT_DATASET_DIR = '/kaggle/input/cmf-pretrain-checkpoints'  # ← EDIT THIS

_CONFIG_CANDIDATES = [
    CKPT_DATASET_DIR,
    f'{CKPT_DATASET_DIR}/checkpoints',
]

config_path = None
for _candidate_root in _CONFIG_CANDIDATES:
    _candidate_config = f'{_candidate_root}/config.json'
    if os.path.exists(_candidate_config):
        config_path = _candidate_config
        CKPT_ROOT = _candidate_root
        break

assert config_path is not None, (
    'config.json not found. Checked:\n' +
    '\n'.join(f'  - {p}/config.json' for p in _CONFIG_CANDIDATES) + '\n' +
    'Make sure Notebook 1 finished and the correct output dataset is attached.')

with open(config_path) as f:
    CFG = json.load(f)

# ── Restore all settings from Notebook 1 ─────────────────────────────
TEST_MODE     = CFG['TEST_MODE']
TEST_FRACTION = CFG['TEST_FRACTION']
_MODE_TAG     = CFG['_MODE_TAG']
DATASET       = CFG['DATASET']
ARCH          = CFG['ARCH']
IS_VIT        = CFG['IS_VIT']
SEED          = CFG['SEED']
NUM_CLASSES        = CFG['NUM_CLASSES']
CLASS_LABEL_NAMES  = CFG.get('CLASS_LABEL_NAMES')  # list of str, or None
ALL_EXPS           = [list(g) for g in CFG['ALL_EXPS']]  # JSON arrays → lists
PRETRAIN_LR   = CFG['PRETRAIN_LR']
PRETRAIN_EPOCHS = CFG['PRETRAIN_EPOCHS']
PRETRAIN_BS   = CFG['PRETRAIN_BS']
PRETRAIN_PATIENCE = CFG['PRETRAIN_PATIENCE']
RETRAIN_EPOCHS = CFG['RETRAIN_EPOCHS']
_TOTAL     = {DATASET: CFG['TOTAL']}
_PER_CLASS = {DATASET: CFG['PER_CLASS']}

# Checkpoint paths — re-root from the original Notebook 1 CKPT_ROOT to
# the detected dataset root inside Kaggle input storage.
_old_root = CFG['CKPT_ROOT']

def _repath(p):
    """Re-root a path saved in config.json from the original CKPT_ROOT to the attached dataset root."""
    return p.replace(_old_root, CKPT_ROOT)

CKPT_PRETRAIN = _repath(CFG['CKPT_PRETRAIN'])
CKPT_CMF_FT   = _repath(CFG['CKPT_CMF_FT'])

# Verify the two key checkpoints exist
for p in [CKPT_PRETRAIN, CKPT_CMF_FT]:
    if os.path.exists(p):
        print(f'  ✓ {p}')
    else:
        print(f'  ✗ MISSING: {p}')

# Data path — download fresh in this session
DATA_PATH = '/kaggle/working/data'
os.makedirs(DATA_PATH, exist_ok=True)

# Working output directory for unlearn checkpoints, eval JSONs, CSVs
WORK_ROOT = '/kaggle/working'

print(f'Loaded config: {config_path}')
print(f'Detected checkpoint root: {CKPT_ROOT}')
print(f'\nMode={"TEST" if TEST_MODE else "FULL"}  Dataset={DATASET}  Arch={ARCH}')
print(f'Forget groups: {len(ALL_EXPS)}  → {ALL_EXPS}')
print(f'_TOTAL={_TOTAL[DATASET]}  _PER_CLASS={_PER_CLASS[DATASET]}')

## B. Method Selection — Group B

Group B methods run in this notebook. Group A runs in `notebook_2_unlearn`.

| Key | Paper name | Type |
|-----|-----------|------|
| `SVD` | SVD (gradient-free) | Standard |
| `random_label_CMF_RemoveFC` | Random Label w. CMF | CMF |
| `salun_CMF_RemoveFC` | SalUn w. CMF | CMF |
| `scrub_CMF_RemoveFC` | SCRUB w. CMF | CMF |
| `tarun_CMF_RemoveFC` | UNSIR w. CMF | CMF |

In [ ]:
# ── Group B methods (this notebook) ────────────────────────────────
RUN_METHODS = [
    'SVD',                       # SVD (gradient-free)
    'random_label_CMF_RemoveFC', # Random Label w. CMF
    'salun_CMF_RemoveFC',        # SalUn w. CMF
    'scrub_CMF_RemoveFC',        # SCRUB w. CMF
    'tarun_CMF_RemoveFC',        # UNSIR w. CMF
]

STANDARD_METHODS = [m for m in RUN_METHODS if 'CMF' not in m]
CMF_METHODS      = [m for m in RUN_METHODS if 'CMF' in m]

print(f'Methods ({len(RUN_METHODS)}): {RUN_METHODS}')
print(f'  Standard : {STANDARD_METHODS}')
print(f'  CMF      : {CMF_METHODS}')
print(f'Total runs : {len(RUN_METHODS)} × {len(ALL_EXPS)} = {len(RUN_METHODS)*len(ALL_EXPS)}')

## C-helpers. Args, Data & Unlearn Helpers

In [ ]:
from utils import get_dataset, get_model, get_retain_forget_partition, test, load_encoder_ckpt_safely
from unlearn import unlear_func
import utils as _utils_module, functools

# Silence verbose=True default inside unlearn functions
_orig_test = test
@functools.wraps(_orig_test)
def test(*a, verbose=False, **kw):
    return _orig_test(*a, verbose=verbose, **kw)
_utils_module.test = test

# ── Unlearning hyperparameters ────────────────────────────────────────
UNLEARN_BS = 8 if TEST_MODE else 128

_LR = {
    'random_label':              {'cifar10':{'s':1e-2,'m':1e-2},'cifar100':{'s':3e-3,'m':3e-3},'tinyimagenet':{'s':5e-4,'m':5e-4}},
    'salun':                     {'cifar10':{'s':1e-2,'m':1e-2},'cifar100':{'s':3e-3,'m':3e-3},'tinyimagenet':{'s':5e-4,'m':5e-4}},
    'grad_ascent_descent':       {'cifar10':{'s':1e-3,'m':1e-3},'cifar100':{'s':5e-5,'m':1e-4},'tinyimagenet':{'s':5e-4,'m':5e-4}},
    'scrub':                     {'cifar10':{'s':1e-4,'m':1e-4},'cifar100':{'s':1e-3,'m':3e-4},'tinyimagenet':{'s':5e-3,'m':1e-3}},
    'tarun':                     {'cifar10':{'s':2e-3,'m':5e-5},'cifar100':{'s':3e-5,'m':3e-5},'tinyimagenet':{'s':2e-5,'m':2e-5}},
    'SVD':                       {'cifar10':{'s':1e-2,'m':1e-2},'cifar100':{'s':1e-2,'m':1e-2},'tinyimagenet':{'s':1e-3,'m':1e-3}},
    'random_label_CMF_RemoveFC': {'cifar10':{'s':1e-4,'m':2e-3},'cifar100':{'s':2e-3,'m':2e-3},'tinyimagenet':{'s':1e-2,'m':1e-2}},
    'salun_CMF_RemoveFC':        {'cifar10':{'s':2e-4,'m':2e-3},'cifar100':{'s':2e-3,'m':2e-3},'tinyimagenet':{'s':1e-2,'m':1e-2}},
    'grad_ascent_descent_CMF_RemoveFC':{'cifar10':{'s':1e-4,'m':1e-4},'cifar100':{'s':1e-4,'m':1e-4},'tinyimagenet':{'s':3e-5,'m':3e-5}},
    'tarun_CMF_RemoveFC':        {'cifar10':{'s':5e-5,'m':5e-5},'cifar100':{'s':5e-5,'m':5e-5},'tinyimagenet':{'s':2e-5,'m':2e-5}},
    'scrub_CMF_RemoveFC':        {'cifar10':{'s':5e-3,'m':1e-3},'cifar100':{'s':5e-3,'m':5e-3},'tinyimagenet':{'s':5e-3,'m':1e-3}},
}
_EPOCHS = {
    'random_label':3,'salun':3,'grad_ascent_descent':3,
    'scrub':3,'tarun':3,'SVD':200,
    'random_label_CMF_RemoveFC':4,'salun_CMF_RemoveFC':4,
    'grad_ascent_descent_CMF_RemoveFC':3,'tarun_CMF_RemoveFC':3,'scrub_CMF_RemoveFC':3,
}
if TEST_MODE:
    _EPOCHS = {k: 1 for k in _EPOCHS}

_SVD = {
    'cifar10':     {'alpha_r':100, 'alpha_f':3,  'samples':900,'max_patches':10000},
    'cifar100':    {'alpha_r':1000,'alpha_f':30, 'samples':990,'max_patches':10000},
    'tinyimagenet':{'alpha_r':30,  'alpha_f':10, 'samples':999,'max_patches':10000},
}
if TEST_MODE:
    _SVD = {k: {'alpha_r':2,'alpha_f':1,'samples':4,'max_patches':10} for k in _SVD}

def get_lr(method, forget_classes):
    mode = 's' if len(forget_classes) == 1 else 'm'
    return _LR.get(method, {}).get(DATASET, {}).get(mode, 1e-3)

def get_epochs(method):
    return _EPOCHS.get(method, 1 if TEST_MODE else 3)

def make_args(**ov):
    d = dict(
        dataset=DATASET, arch=ARCH, data_path=DATA_PATH,
        num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
        batch_size=PRETRAIN_BS, test_batch_size=256,
        epochs_or_steps=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR, momentum=0.9, weight_decay=5e-4, gamma=0.5,
        seed=SEED, log_interval=200, val_ratio=0.1,
        patience=PRETRAIN_PATIENCE, warmup_epochs=5,
        min_lr=1e-5, lr_scheduler='cosine',
        unlearn_method='pre_train', unlearn_class=[],
        num_retain_samples=_TOTAL[DATASET], num_forget_samples=0,
        grad_norm_clip=None, salun_threshold=0.5,
        goel_exact=False, ssd_lambda=1, ssd_alpha=10,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=3,
        SVD_alpha_r=100, SVD_alpha_f=3, SVD_samples=900, SVD_max_patches=10000,
        tarun_impair_lr=2e-4, tarun_samples_per_class=1000,
        no_cuda=False, no_mps=True, dry_run=False,
        save_model=True, save_path=None,
        sub_set_mode=False, sub_set_samples=10000,
        no_train_transform=False, train_transform=True,
        gpu_id=0, multiclass=False, class_names=None,
        do_mia=False, do_mia_ulira=False, plot_mia_roc=False,
        prob_batch_size=128,
        freeze_except_last=False, zero_last_layer=False,
        remove_FC=False, CMF_momentum=0.9, CMFClassifier=True,
        do_lp=False, lp_every=0, ncc_every=0,
        eval_every_iter=0, lp_every_iter=0, ncc_every_iter=0,
        pretrained=IS_VIT,
        project_name='kaggle', group_name='paper_repro',
    )
    d.update(ov)
    return argparse.Namespace(**d)

print('Helpers loaded.')

In [ ]:
args_base = make_args()
dataset_train, dataset_test = get_dataset(args_base)
CLASS_LABEL_NAMES = args_base.class_label_names
print(f'Full dataset — Train={len(dataset_train)}  Test={len(dataset_test)}  Classes={NUM_CLASSES}')

# Re-apply the same stratified shrink that Notebook 1 used in TEST_MODE
if TEST_MODE:
    def _stratified_subset(ds, fraction, seed=SEED):
        labels = (ds.targets if hasattr(ds, 'targets')
                  else [ds.dataset.targets[i] for i in ds.indices]
                  if hasattr(ds, 'indices') else [s[1] for s in ds.samples])
        rng = random.Random(seed)
        by_class = collections.defaultdict(list)
        for idx, lbl in enumerate(labels):
            by_class[int(lbl)].append(idx)
        kept = []
        for cls_idx in sorted(by_class):
            cls_pool = by_class[cls_idx]
            rng.shuffle(cls_pool)
            n_keep = max(1, math.ceil(len(cls_pool) * fraction))
            kept.extend(cls_pool[:n_keep])
        sub = torch.utils.data.Subset(ds, kept)
        base_targets = ds.targets if hasattr(ds, 'targets') else [
            ds.dataset.targets[i] for i in ds.indices]
        sub.targets = [base_targets[i] for i in kept]
        return sub

    dataset_train = _stratified_subset(dataset_train, TEST_FRACTION)
    dataset_test  = _stratified_subset(dataset_test,  TEST_FRACTION)
    print(f'TEST_MODE: {TEST_FRACTION*100:.1f}% → '
          f'Train={len(dataset_train)}  Test={len(dataset_test)}')
else:
    print(f'Train={len(dataset_train)}  Test={len(dataset_test)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

LOADER_KW = dict(batch_size=PRETRAIN_BS, num_workers=2, pin_memory=True,
                 shuffle=True, drop_last=True)
TEST_KW   = dict(batch_size=min(256, len(dataset_test)), num_workers=2,
                 pin_memory=True, shuffle=False)
train_loader = torch.utils.data.DataLoader(dataset_train, **LOADER_KW)
test_loader  = torch.utils.data.DataLoader(dataset_test,  **TEST_KW)

# Load original model for reference / t-SNE
args_pt   = make_args(unlearn_method='pre_train', epochs_or_steps=PRETRAIN_EPOCHS,
                      lr=PRETRAIN_LR, patience=PRETRAIN_PATIENCE)
orig_model = get_model(args_pt, device)
orig_model.load_state_dict(torch.load(CKPT_PRETRAIN, map_location=device))
orig_model.eval()
print('\n── Original model test accuracy ──')
test(orig_model, device, test_loader, [], CLASS_LABEL_NAMES, NUM_CLASSES,
     plot_cm=False, job_name='original', set_name='Test')

## C. Run All Unlearning Methods

In [ ]:
def run_unlearn(method, forget_classes, start_ckpt, extra_args=None):
    """
    Run one unlearning experiment.
    Saves checkpoint to /kaggle/working/checkpoints/<method>/...
    Returns (retain_acc, forget_acc, model).
    """
    n_forget   = len(forget_classes)
    num_forget = n_forget * _PER_CLASS[DATASET]
    num_retain = _TOTAL[DATASET] - num_forget
    forget_str = ','.join(str(c) for c in forget_classes)
    lr         = get_lr(method, forget_classes)
    epochs     = get_epochs(method)
    is_cmf     = 'CMF' in method

    kw = dict(
        unlearn_method=method, epochs_or_steps=epochs,
        lr=lr, batch_size=UNLEARN_BS,
        num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
        num_retain_samples=num_retain, num_forget_samples=num_forget,
        unlearn_class=list(forget_classes),
        remove_FC=is_cmf, CMFClassifier=is_cmf,
        grad_norm_clip=1.0, lp_every=0, ncc_every=0,
    )
    if method in ('salun','salun_CMF_RemoveFC'):
        kw['salun_threshold'] = 0.5
    if method == 'SVD':
        s = _SVD[DATASET]
        kw.update(SVD_alpha_r=s['alpha_r'], SVD_alpha_f=s['alpha_f'],
                  SVD_samples=s['samples'], SVD_max_patches=s['max_patches'])
    if method in ('tarun','tarun_CMF_RemoveFC'):
        kw.update(tarun_impair_lr=1e-4 if DATASET=='cifar10' else 2e-4,
                  tarun_samples_per_class=1000)
    if method in ('scrub','scrub_CMF_RemoveFC'):
        kw.update(scrub_del_bsz=64, scrub_sgda_bsz=64,
                  scrub_msteps=2, scrub_epochs=epochs)
    if extra_args:
        kw.update(extra_args)

    args = make_args(**kw)

    retain_ds, forget_ds = get_retain_forget_partition(args, dataset_train, forget_classes)
    retain_loader = torch.utils.data.DataLoader(retain_ds, **LOADER_KW)
    forget_loader = (torch.utils.data.DataLoader(forget_ds, **LOADER_KW)
                     if len(forget_ds) > 0 else None)
    _, test_forget_ds = get_retain_forget_partition(args, dataset_test, forget_classes)
    tf_loader = torch.utils.data.DataLoader(test_forget_ds, **TEST_KW)

    m = get_model(args, device)
    m.load_state_dict(torch.load(start_ckpt, map_location=device))
    optimizer = optim.SGD(m.parameters(), lr=lr,
                          momentum=0.9, weight_decay=5e-4, nesterov=True)

    print(f'\n{"-"*60}')
    print(f'  METHOD : {method}')
    print(f'  forget : {forget_classes}   lr={lr}   epochs={epochs}')
    print(f'{"-"*60}')

    unlearnt = unlear_func[method](
        args=args, model=m, device=device,
        retain_loader=retain_loader, forget_loader=forget_loader,
        train_loader=train_loader, val_loader=None, test_loader=test_loader,
        optimizer=optimizer, epochs=epochs,
        train_dataset=dataset_train, val_index=np.arange(len(dataset_train)),
        test_forget_loader=tf_loader,
    )

    # Save to /kaggle/working/checkpoints/ (writable)
    ckpt_out = f'/kaggle/working/checkpoints/{method}/{DATASET}_{ARCH}_{_MODE_TAG}/{forget_str}.pt'
    os.makedirs(os.path.dirname(ckpt_out), exist_ok=True)
    torch.save(unlearnt.state_dict(), ckpt_out)

    unlearnt.eval()
    ra, fa, _ = test(unlearnt, device, test_loader, forget_classes,
                     CLASS_LABEL_NAMES, NUM_CLASSES,
                     plot_cm=False, job_name=method, set_name='Test', verbose=False)
    print(f'  → retain={ra:.4f}  forget={fa:.4f}')
    return ra, fa, unlearnt

print('run_unlearn helper defined.')

In [ ]:
# ── Standard unlearning methods (Table 1) ─────────────────────────────
RESULTS = []

if not STANDARD_METHODS:
    print('No standard methods selected — skipping.')
else:
    print(f'Running {len(STANDARD_METHODS)} standard method(s): {STANDARD_METHODS}')

for forget_classes in ALL_EXPS:
    forget_str = ','.join(str(c) for c in forget_classes)
    if not STANDARD_METHODS:
        break
    print(f'\n>>> forget={forget_classes}')
    for method in STANDARD_METHODS:
        try:
            ra, fa, _ = run_unlearn(method, forget_classes, CKPT_PRETRAIN)
            RESULTS.append(dict(forget=forget_str, method=method,
                                retain_acc=ra, forget_acc=fa))
        except Exception as e:
            print(f'  ERROR {method}: {e}')
            RESULTS.append(dict(forget=forget_str, method=method,
                                retain_acc=float('nan'), forget_acc=float('nan')))

print('\nStandard methods done.')

In [ ]:
# ── CMF-enhanced unlearning methods (Table 3) ─────────────────────────
RESULTS = [r for r in RESULTS if r['method'] not in CMF_METHODS]

if not CMF_METHODS:
    print('No CMF methods selected — skipping.')
else:
    print(f'Running {len(CMF_METHODS)} CMF method(s): {CMF_METHODS}')

for forget_classes in ALL_EXPS:
    forget_str = ','.join(str(c) for c in forget_classes)
    if not CMF_METHODS:
        break
    print(f'\n>>> forget={forget_classes}')
    for method in CMF_METHODS:
        try:
            ra, fa, _ = run_unlearn(method, forget_classes, CKPT_CMF_FT)
            RESULTS.append(dict(forget=forget_str, method=method,
                                retain_acc=ra, forget_acc=fa))
        except Exception as e:
            print(f'  ERROR {method}: {e}')
            RESULTS.append(dict(forget=forget_str, method=method,
                                retain_acc=float('nan'), forget_acc=float('nan')))

print('\nCMF methods done.')

## D. Evaluation — Output, Linear Probe, NCC (Table 1 & 3)

`evaluation.py` writes a JSON file per (method, forget_group).  
We pass `--save-model` to enable the write, then read the JSON directly.

In [ ]:
# Retrain checkpoints come from the read-only dataset; unlearn checkpoints
# were just written to /kaggle/working/checkpoints/
ALL_METHODS  = list(dict.fromkeys(RUN_METHODS))
EVAL_RESULTS = []

def _find_eval_json(method, forget_str):
    direct = f'/kaggle/working/evaluations/{method}/{DATASET}_{ARCH}/{forget_str}.json'
    if os.path.exists(direct):
        return direct
    matches = sorted(glob.glob(
        f'/kaggle/working/evaluations/{method}*/{DATASET}_{ARCH}/{forget_str}.json'
    ))
    return matches[0] if matches else direct

def _read_eval_json(json_path):
    """Read the structured JSON written by evaluation.py."""
    try:
        with open(json_path) as f:
            raw = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {}
    res = {
        'out_retain': raw.get('test_retain_acc'),
        'out_forget': raw.get('test_forget_acc'),
        'lp_retain' : raw.get('lp_acc_test_retain'),
        'lp_forget' : raw.get('lp_acc_test_forget'),
        'ncc_retain': raw.get('ncc_acc_test_retain'),
        'ncc_forget': raw.get('ncc_acc_test_forget'),
    }
    return {k: v for k, v in res.items() if v is not None}

for forget_classes in ALL_EXPS:
    n_forget   = len(forget_classes)
    num_forget = n_forget * _PER_CLASS[DATASET]
    num_retain = _TOTAL[DATASET] - num_forget
    forget_str = ','.join(str(c) for c in forget_classes)

    for method in ALL_METHODS:
        # Unlearn checkpoints are in writable /kaggle/working/checkpoints/
        ckpt = f'/kaggle/working/checkpoints/{method}/{DATASET}_{ARCH}_{_MODE_TAG}/{forget_str}.pt'
        if not os.path.exists(ckpt):
            continue
        is_cmf_exp = 'CMF' in method

        json_path = _find_eval_json(method, forget_str)
        logfile   = f'/kaggle/working/evaluations/{method}/{DATASET}_{ARCH}/{forget_str.replace(",","_")}.log'
        os.makedirs(os.path.dirname(logfile), exist_ok=True)

        cmd = (
            f'python {REPO_DIR}/evaluation.py'
            f' --dataset {DATASET}'
            f' --arch {ARCH}'
            f' --data-path {DATA_PATH}'
            f' --unlearn-method {method}'
            f' --save-path "{ckpt}"'
            f' --epochs-or-steps {1 if TEST_MODE else 10}'
            f' --batch-size {8 if TEST_MODE else 128}'
            f' --lr 3e-4'
            f' --num-retain-samples {num_retain}'
            f' --num-forget-samples {num_forget}'
            f' --unlearn-class "{forget_str}"'
            f' --prob-batch-size 128'
            f' --do-linear-probe'
            f' --use-last-only'
            f' --do-ncc-mismatch'
            f' --save-model'
            + (' --remove_FC' if is_cmf_exp else '')
            + (' --pretrained' if IS_VIT else '')
            + f' > {logfile} 2>&1'
        )
        print(f'\n[{method}]  forget={forget_classes}')
        rc = sh(cmd, verbose=False)
        if rc != 0:
            print(f'  ✗ failed (exit {rc}) — see {logfile}')
        else:
            json_path = _find_eval_json(method, forget_str)
            parsed = _read_eval_json(json_path)
            if parsed:
                _f = lambda v: f'{v:7.4f}' if isinstance(v, float) else str(v)
                print(f'  Output  retain={_f(parsed.get("out_retain","?"))}  '
                      f'forget={_f(parsed.get("out_forget","?"))}')
                if 'lp_retain' in parsed:
                    print(f'  LP      retain={parsed["lp_retain"]:7.4f}  forget={parsed["lp_forget"]:7.4f}')
                if 'ncc_retain' in parsed:
                    print(f'  NCC     retain={parsed["ncc_retain"]:7.4f}  forget={parsed["ncc_forget"]:7.4f}')
                EVAL_RESULTS.append(dict(
                    method=method, forget=forget_str,
                    out_retain=parsed.get('out_retain'), out_forget=parsed.get('out_forget'),
                    lp_retain=parsed.get('lp_retain'),  lp_forget=parsed.get('lp_forget'),
                    ncc_retain=parsed.get('ncc_retain'), ncc_forget=parsed.get('ncc_forget'),
                ))
            else:
                print(f'  ✓ done but no readable JSON at {json_path} — see {logfile}')

if EVAL_RESULTS:
    eval_df = pd.DataFrame(EVAL_RESULTS)
    _cols = [c for c in ['out_retain','out_forget','lp_retain',
                          'lp_forget','ncc_retain','ncc_forget']
             if c in eval_df.columns and eval_df[c].notna().any()]
    summary_eval = eval_df.groupby('method')[_cols].mean().round(4)
    print('\n=== Evaluation Summary (mean over forget groups) ===')
    print(summary_eval.to_string())

print('\nEvaluation complete.')

## E. Results Summary Table (Table 1 & 3 of paper)

In [ ]:
# ── Load retrain results from Notebook 1 CSV ──────────────────────────
rt_csv = f'{CKPT_ROOT}/retrain_results.csv'
if os.path.exists(rt_csv):
    rt_df = pd.read_csv(rt_csv)
    rt_df['method'] = 'Retrain'
    rt_df = rt_df.rename(columns={'retain_acc':'retain_acc','forget_acc':'forget_acc'})
    print(f'Loaded {len(rt_df)} retrain rows from {rt_csv}')
else:
    print(f'WARNING: retrain_results.csv not found at {rt_csv} — Retrain row will be missing.')
    rt_df = pd.DataFrame(columns=['forget','method','retain_acc','forget_acc'])

# ── Original model accuracy per forget group ──────────────────────────
orig_rows = []
for forget_classes in ALL_EXPS:
    forget_str = ','.join(str(c) for c in forget_classes)
    ra, fa, _ = test(orig_model, device, test_loader, forget_classes,
                     CLASS_LABEL_NAMES, NUM_CLASSES,
                     plot_cm=False, job_name='original', set_name='Test')
    orig_rows.append(dict(forget=forget_str, method='Original',
                          retain_acc=ra, forget_acc=fa))

# ── Combine all output-level results ─────────────────────────────────
unlearn_df = pd.DataFrame(RESULTS)
df = pd.concat([pd.DataFrame(orig_rows), rt_df, unlearn_df], ignore_index=True)

summary = df.groupby('method')[['retain_acc','forget_acc']].mean()
summary.columns = ['Mean Retain Acc', 'Mean Forget Acc']
summary = summary.round(4)

ROW_ORDER = [
    'Original','Retrain',
    'grad_ascent_descent','random_label','salun','scrub','tarun','SVD',
    'random_label_CMF_RemoveFC','salun_CMF_RemoveFC',
    'grad_ascent_descent_CMF_RemoveFC','scrub_CMF_RemoveFC','tarun_CMF_RemoveFC',
]
present = [r for r in ROW_ORDER if r in summary.index]
summary = summary.loc[present]

NAME_MAP = {
    'Original':'Original','Retrain':'Retain-only Retrain',
    'grad_ascent_descent':'NegGrad+','random_label':'Random Label',
    'salun':'SalUn','scrub':'SCRUB','tarun':'UNSIR','SVD':'SVD',
    'random_label_CMF_RemoveFC':'Random Label w. CMF',
    'salun_CMF_RemoveFC':'SalUn w. CMF',
    'grad_ascent_descent_CMF_RemoveFC':'NegGrad+ w. CMF',
    'scrub_CMF_RemoveFC':'SCRUB w. CMF',
    'tarun_CMF_RemoveFC':'UNSIR w. CMF',
}
summary.index = [NAME_MAP.get(i, i) for i in summary.index]

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 30)
print(f'\n=== Output-Level Results — {DATASET}/{ARCH} '
      f'(mean over {len(ALL_EXPS)} forget groups) ===')
print(summary.to_string())

csv_path = f'/kaggle/working/results_{DATASET}_{ARCH}.csv'
df.to_csv(csv_path, index=False)
print(f'\nFull results saved: {csv_path}')

## F. Bar Chart — Retain vs Forget Accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
labels = list(summary.index)
x = np.arange(len(labels))
w = 0.38

b1 = ax.bar(x - w/2, summary['Mean Retain Acc'], w,
            label='Retain Acc', color='steelblue', alpha=0.85)
b2 = ax.bar(x + w/2, summary['Mean Forget Acc'], w,
            label='Forget Acc', color='tomato', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.08)
ax.set_title(f'Output-Level Accuracy — {DATASET}/{ARCH}\n'
             f'(mean over {len(ALL_EXPS)} forget groups)')
ax.legend()

_std_keys = {'grad_ascent_descent','random_label','salun','scrub','tarun','SVD'}
_cmf_keys = {k for k in ROW_ORDER if 'CMF' in k}
_n_base = sum(1 for m in present if m in ('Original','Retrain'))
_n_std  = sum(1 for m in present if m in _std_keys)
_n_cmf  = sum(1 for m in present if m in _cmf_keys)
if _n_base > 0 and _n_std > 0:
    ax.axvline(x=_n_base - 0.5, color='grey', linewidth=0.8, linestyle='--')
    ax.text(_n_base/2 - 0.5, 1.03, 'Baselines', ha='center', fontsize=8)
if _n_std > 0 and _n_cmf > 0:
    ax.axvline(x=_n_base + _n_std - 0.5, color='grey', linewidth=0.8, linestyle='--')
    ax.text(_n_base + _n_std/2 - 0.5, 1.03, 'Standard Methods', ha='center', fontsize=8)
    ax.text(_n_base + _n_std + _n_cmf/2 - 0.5, 1.03, 'CMF Methods', ha='center', fontsize=8)
elif _n_std > 0:
    ax.text(_n_base + _n_std/2 - 0.5, 1.03, 'Standard Methods', ha='center', fontsize=8)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=6.5)

plt.tight_layout()
fig_path = f'/kaggle/working/results_chart_{DATASET}_{ARCH}.png'
plt.savefig(fig_path, dpi=130)
plt.show()
print('Chart saved:', fig_path)

## G. t-SNE Visualisation (Figure 3 & 6 of paper)

Compares feature-space distributions: Original vs. one standard method vs. one CMF method.  
The forgotten class should be linearly separable in the standard method (Figure 3)  
and merged with retain classes in the CMF method (Figure 6).

In [ ]:
from sklearn.manifold import TSNE

def collect_features(model, loader, max_pts=3000):
    model.eval()
    feats, labs = [], []
    seen = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            if hasattr(model, 'encoder'):
                z = model.encoder(x)
            elif hasattr(model, 'feature'):
                z = model.feature(x)
            else:
                hooks, buf = [], []
                def _hook(mod, inp, out):
                    buf.append(out.detach().cpu().flatten(1))
                for name, mod in reversed(list(model.named_modules())):
                    if isinstance(mod, (nn.AdaptiveAvgPool2d, nn.LayerNorm)):
                        hooks.append(mod.register_forward_hook(_hook))
                        break
                model(x)
                for h in hooks: h.remove()
                z = buf[0] if buf else model(x).detach().cpu()
            feats.append(z.cpu().numpy() if isinstance(z, torch.Tensor) else z)
            labs.append(y.numpy())
            seen += x.size(0)
            if seen >= max_pts:
                break
    return np.concatenate(feats)[:max_pts], np.concatenate(labs)[:max_pts]

def plot_tsne(model, loader, title, forget_cls, max_pts=2000, ax=None):
    feats, labs = collect_features(model, loader, max_pts)
    emb = TSNE(n_components=2, perplexity=30, random_state=0,
               max_iter=500).fit_transform(feats)
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    for c in sorted(set(labs)):
        mask = labs == c
        color  = 'red' if c in forget_cls else None
        zorder = 5   if c in forget_cls else 1
        ax.scatter(emb[mask, 0], emb[mask, 1], s=6, alpha=0.5,
                   color=color, zorder=zorder,
                   label=f'cls {c}' if c in forget_cls else None)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    return ax

def _load_ckpt_into_model(model, ckpt_path, device):
    sd = torch.load(ckpt_path, map_location=device)
    if all(k.startswith('encoder.') for k in sd if not k.startswith('CMFweights')):
        sd = {k[len('encoder.'):]: v for k, v in sd.items() if k.startswith('encoder.')}
    model.load_state_dict(sd, strict=False)

vis_forget     = ALL_EXPS[0]
vis_forget_str = ','.join(str(c) for c in vis_forget)
vis_loader     = torch.utils.data.DataLoader(dataset_test, batch_size=256,
                                              shuffle=True, num_workers=2)

vis_models = [('Original', CKPT_PRETRAIN, False)]
for mth in STANDARD_METHODS:
    ck = f'/kaggle/working/checkpoints/{mth}/{DATASET}_{ARCH}_{_MODE_TAG}/{vis_forget_str}.pt'
    if os.path.exists(ck):
        vis_models.append((mth, ck, False))
        break
for mth in CMF_METHODS:
    ck = f'/kaggle/working/checkpoints/{mth}/{DATASET}_{ARCH}_{_MODE_TAG}/{vis_forget_str}.pt'
    if os.path.exists(ck):
        vis_models.append((mth, ck, True))
        break

fig, axes = plt.subplots(1, len(vis_models), figsize=(5*len(vis_models), 4.5))
if len(vis_models) == 1:
    axes = [axes]

for ax, (name, ck, is_cmf) in zip(axes, vis_models):
    a_pt = make_args(unlearn_class=list(vis_forget), remove_FC=is_cmf)
    m = get_model(a_pt, device)
    _load_ckpt_into_model(m, ck, device)
    m.eval()
    label = name.replace('_CMF_RemoveFC', ' w. CMF').replace('_', ' ')
    plot_tsne(m, vis_loader, label, vis_forget, max_pts=2000, ax=ax)
    ax.legend(loc='lower right', fontsize=7, markerscale=2)

plt.suptitle(f't-SNE — {DATASET}/{ARCH} | forget: {vis_forget}  (red = forgotten)',
             fontsize=11)
plt.tight_layout()
tsne_path = f'/kaggle/working/tsne_{DATASET}_{ARCH}.png'
plt.savefig(tsne_path, dpi=130)
plt.show()
print('t-SNE saved:', tsne_path)